In [ ]:

import sys
from collections import Counter
from pathlib import Path

# find the repo root (the folder containing `src`) no matter where Jupyter was started
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
from src.data.enron import load_enron
from src.data.splits import make_splits
from src.features.extractor import extract_frame, FEATURES
from src.profiling.stats import icc1

ENRON_DIR = ROOT / "data" / "external" / "Enron"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)

stats = Counter()
raw = load_enron(ENRON_DIR, stats=stats)
print(dict(stats))
if raw.empty:
    raise RuntimeError(
        "Files were found but no message survived cleaning. Look at the counts above: "
        "mostly 'rejected_bad_header_or_date' means the Date header could not be parsed; "
        "mostly 'rejected_too_short_or_pure_forward' means the cleaning is too strict. "
        "Send me the counts plus the first 15 lines of one raw email."
    )
print(f"{len(raw):,} cleaned messages from {raw.user_id.nunique()} mailboxes")

for MIN_MSGS in (100, 50, 30, 20):
    if (raw.groupby("user_id").size() >= MIN_MSGS).sum() >= 10:
        break
print(f"MIN_MSGS used: {MIN_MSGS}")
df = make_splits(raw, min_msgs=MIN_MSGS)
if df.empty:
    raise RuntimeError("No user has enough messages. Try a smaller MIN_MSGS or check the load report.")
df = extract_frame(df)
df.to_csv(OUT_DIR / "enron_messages.csv", index=False)
print(f"{len(df):,} messages from {df.user_id.nunique()} users saved to {OUT_DIR / 'enron_messages.csv'}")

per_user = df.groupby("user_id").agg(n=("message", "size"),
                                     first=("timestamp", "min"), last=("timestamp", "max"),
                                     n_recipients_seen=("recipient_id", "nunique"))
print(per_user["n"].describe().round(1))
print("share with quoted context:", round((df["context"].str.len() > 0).mean(), 3))
print(df["user_split"].value_counts().to_dict(), df["time_split"].value_counts().to_dict())
per_user.head()

nz = (df[FEATURES] != 0).mean().round(3).rename("nonzero_rate")
nz

KEY = ["log_words", "avg_sent_len", "punct_density", "n_questions", "n_exclam",
       "caps_ratio", "starts_lower", "has_newline", "has_list"]
pair = df[df.recipient_id != ""].copy()
pair["pair"] = pair["user_id"] + "_" + pair["recipient_id"]
pair = pair[pair.groupby("pair")["pair"].transform("size") >= 3]
res = pd.DataFrame({
    "user_ICC": {f: icc1(df[f], df["user_id"]) for f in KEY},
    "user_x_recipient_ICC": {f: icc1(pair[f], pair["pair"]) if len(pair) else np.nan for f in KEY},
}).round(3)
print(f"user x recipient pairs analysed: {pair['pair'].nunique()}")
res

has_ctx = df[df["context"].str.len() > 0].copy()
if len(has_ctx) > 30:
    has_ctx["ctx_log_words"] = np.log1p(has_ctx["context"].str.split().str.len())
    print("spearman(reply_len, context_len):",
          round(has_ctx[["log_words", "ctx_log_words"]].corr(method="spearman").iloc[0, 1], 3))
else:
    print("Not enough replies with quoted context to compute this.")

df[["message", "context"]].sample(10, random_state=0)

repo root: c:\Dev\PickyTalker
Using CSV file: C:\Dev\PickyTalker\data\external\Enron\emails.csv
  ...20,000 sent messages read
  ...40,000 sent messages read
  ...60,000 sent messages read
  ...80,000 sent messages read
  ...100,000 sent messages read
  ...120,000 sent messages read
{'sent_files_seen': 126058, 'parsed_ok': 102928, 'rejected_too_short_or_pure_forward': 21288, 'rejected_too_long': 1842}
70,116 cleaned messages from 148 mailboxes
MIN_MSGS used: 100
67,705 messages from 96 users saved to c:\Dev\PickyTalker\data\processed\enron_messages.csv
count      96.0
mean      705.3
std       823.8
min       103.0
25%       205.8
50%       410.0
75%       814.5
max      4160.0
Name: n, dtype: float64
share with quoted context: 0.389
{'train': 43158, 'val': 12629, 'test': 11918} {'history': 54201, 'eval': 13504}
user x recipient pairs analysed: 5101
spearman(reply_len, context_len): 0.081


,message,context
29300,"$50,000 is pre-match, so I guess the ultimate ...",
66703,Did you see Tim?,"From: Bump, Dan J. Sent: Wednesday, November 1..."
35576,what is the latest\n\nHunter,
62697,"Martin\n\nPlease, refer John directly to Jinba...",07:16 AM ---------------------------\n\nJinbae...
12044,for meeting file,AM ---------------------------\n\nCindy Olson@...
12794,"Kate, We are missing an ID-S from ST-WBOM to S...",
53932,We bought the gas from Koch Energy Trading (De...,
4677,"Hey, I'm not going to the party tonight. My W...",
63875,Let me know if you need anything else.\n\nStacey,"From: \tReeves, Kathy Sent:\tWednesday, Januar..."
39669,We will propose to Stan and see where we are a...,"From: \tRatner, Michael Sent:\tMonday, Novembe..."
